In [ ]:
# @title Mise en place de l'environnement
!pip install -q "evidently>=0.7,<0.8" "pandas<3"
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames
import random
import warnings

import nltk
import pandas
import sklearn.feature_extraction.text
import sklearn.linear_model
import sklearn.model_selection
import sklearn.pipeline

warnings.filterwarnings("ignore")

nltk.download("words")
nltk.download("wordnet")
nltk.download("omw-1.4")


def pretraite(fichier_entrainement, fichier_test):
  X_entrainement = pandas.read_csv(fichier_entrainement, index_col="Id")
  X_evaluation = pandas.read_csv(fichier_test, index_col="Id")

  y_entrainement = X_entrainement.pop("SalePrice")

  X_tout = pandas.concat([X_entrainement, X_evaluation])

  colonnes_1 = ["LotFrontage"]
  X_tout[colonnes_1] = X_tout[colonnes_1].fillna(X_entrainement[colonnes_1].median())

  colonnes_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  X_tout[colonnes_2] = X_tout[colonnes_2].fillna(
    X_entrainement[colonnes_2].mode().iloc[0, :]
  )

  colonnes_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  X_tout[colonnes_4] = X_tout[colonnes_4].fillna(0)

  colonnes_5 = ["Functional"]
  X_tout[colonnes_5] = X_tout[colonnes_5].fillna("Typ")

  X_tout = X_tout.fillna("NA")

  colonnes_vers_texte = ["MSSubClass"]
  X_tout[colonnes_vers_texte] = X_tout[colonnes_vers_texte].astype(str)

  correspondance_qualite = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  colonnes_qualite = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  correspondance_revetement = dict(NA=0, Grvl=1, Pave=2)
  correspondance_finition = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  remplacements = dict(
    Alley=correspondance_revetement,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=correspondance_finition,
    BsmtFinType2=correspondance_finition,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for colonne_qualite in colonnes_qualite:
    remplacements[colonne_qualite] = correspondance_qualite

  X_tout.replace(remplacements, inplace=True)

  print(f"Nombre de valeurs manquantes : {X_tout.isnull().sum().sum()}")

  return (
    X_tout.iloc[: X_entrainement.shape[0], :],
    y_entrainement,
    X_tout.iloc[X_entrainement.shape[0] :, :],
  )


def telecharge_avis_medicaments() -> tuple[pandas.DataFrame, pandas.DataFrame]:
  """Data source: https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29

  Citation:
    Felix Gräßer, Surya Kallumadi, Hagen Malberg, and Sebastian Zaunseder.
    2018.
    Aspect-Based Sentiment Analysis of Drug Reviews Applying Cross-Domain and Cross-Data Learning.
    In Proceedings of the 2018 International Conference on Digital Health (DH '18).
    ACM, New York, NY, USA, 121-125.
  """
  donnees_brutes = pandas.read_csv(
    "https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_raw.tsv",
    sep="\t",
    index_col=0,
  )
  return donnees_brutes[["drugName", "condition", "review", "rating"]]


def filtre_avis_medicaments(df: pandas.DataFrame, condition: str) -> pandas.DataFrame:
  df = df.loc[
    (df["condition"] == condition) & (df["rating"].isin([1, 10])),
    ["review", "rating"],
  ]
  df["est_positif"] = df["rating"].apply(lambda x: 0 if x == 1 else 1)
  return df.drop(columns="rating")


def separe_avis_medicaments(
  df: pandas.DataFrame,
) -> tuple[pandas.DataFrame, pandas.DataFrame]:
  X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    df["review"], df["est_positif"], test_size=0.4, random_state=42, shuffle=True
  )

  reference = pandas.DataFrame({"review": X_train, "est_positif": y_train})
  validation = pandas.DataFrame({"review": X_test, "est_positif": y_test})
  return reference, validation

# Monitoring — Dérive des données et des concepts

## Prise en main de la bibliothèque `evidently`

### Chargement des données

In [ ]:
reference, _, actuel = pretraite("ames/train.csv", "ames/test.csv")

### Imports

In [ ]:
from evidently import (
  BinaryClassification,
  DataDefinition,
  Dataset,
  Report,
)
from evidently.descriptors import (
  NonLetterCharacterPercentage,
  OOVWordsPercentage,
  SentenceCount,
  TextLength,
  WordCount,
)
from evidently.metrics import DriftedColumnsCount, ValueDrift
from evidently.presets import (
  ClassificationPreset,
  DataDriftPreset,
)
from evidently.tests import lt, lte
from IPython.display import display

### Rapports

Evidently produit des rapports, auxquels on peut attacher des tests.

Un rapport se lit : un humain y étudie les distributions et les scores de chaque colonne. Un test est une condition vérifiée automatiquement sur une métrique du rapport, par exemple pour arrêter une étape d'un pipeline ou déclencher un réentraînement.

Commencez par un rapport de dérive des données, avec le preset `DataDriftPreset` et ses paramètres par défaut, qui compare `actuel` à `reference` : `rapport.run` prend les `DataFrame` tels quels. Combien de colonnes dérivent ? Quelle méthode Evidently a-t-il choisie pour `GrLivArea`, et pourquoi ?

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/report
- https://docs.evidentlyai.com/metrics/preset_data_drift

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `Report([...])` : une liste de métriques ou de presets
- `rapport.run(current_data=..., reference_data=...)` : les données récentes et la référence, nommées pour ne pas les inverser ; il renvoie un snapshot, que le notebook affiche s'il est la dernière expression de la cellule
- le tableau du rapport donne, pour chaque colonne, la méthode employée et son score ; `GrLivArea` est numérique, et `reference` compte 1 460 lignes (voir la diapositive « Détecter une dérive — Tests statistiques »)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rapport = Report([...])
instantane = rapport.run(current_data=..., reference_data=...)
instantane
```

#### Solution

In [ ]:
rapport = Report([DataDriftPreset()])
# Aucune colonne ne dérive. Pour GrLivArea, numérique, Evidently a choisi la
# distance de Wasserstein normalisée (seuil 0,1) et non un test de KS : au-delà
# de 1 000 lignes de référence (ici 1 460), une p-valeur signalerait des écarts
# sans importance.
instantane = rapport.run(current_data=actuel, reference_data=reference)
instantane

### Comparer les méthodes de détection

Les méthodes de détection ne donnent pas toutes le même genre de score. Calculez la dérive de `GrLivArea` avec la métrique `ValueDrift` et chacune des méthodes `ks`, `wasserstein` et `psi`, puis affichez pour chacune son score et son seuil. Comparez-les : pourquoi une $p$-valeur et une distance ne se lisent-elles pas dans le même sens ?

**Documentation utile**

- https://docs.evidentlyai.com/metrics/all_metrics
- https://docs.evidentlyai.com/metrics/customize_data_drift

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `ValueDrift(column="GrLivArea", method="ks")` : une métrique par méthode, toutes dans le même `Report`
- `instantane.dict()["metrics"]` : une entrée par métrique, avec son score (`value`) et, dans `config`, sa méthode (`method`) et son seuil (`threshold`)
- la diapositive « Détecter une dérive — Tests statistiques » dit, pour chaque méthode, quand il y a dérive

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rapport = Report([ValueDrift(column="GrLivArea", method=methode) for methode in [...]])
instantane = rapport.run(current_data=actuel, reference_data=reference)
for metrique in instantane.dict()["metrics"]:
  print(metrique["config"]["method"], ..., ...)
```

#### Solution

In [ ]:
rapport = Report(
  [
    ValueDrift(column="GrLivArea", method=methode)
    for methode in ["ks", "wasserstein", "psi"]
  ]
)
instantane = rapport.run(current_data=actuel, reference_data=reference)
for metrique in instantane.dict()["metrics"]:
  methode, seuil = metrique["config"]["method"], metrique["config"]["threshold"]
  print(f"{methode} : score {metrique['value']:.3f}, seuil {seuil}")
# KS donne une p-valeur : la probabilité d'observer un écart au moins aussi grand
# si les deux jeux suivaient la même distribution. Petite, elle signale une dérive
# (sous 0,05) ; ici 0,08. Wasserstein (0,07) et PSI (0,02) donnent une distance :
# grande, elle signale une dérive (à partir de 0,1). Aucune des trois ne conclut
# à une dérive.

### Configurer et sauvegarder un rapport

Par défaut, `DataDriftPreset` choisit la méthode de chaque colonne d'après son type et le nombre de lignes, et déclare le jeu en dérive quand la moitié de ses colonnes dérivent. Configurez-le : la méthode `psi` pour toutes les colonnes, et un jeu en dérive dès que 30 % de ses colonnes dérivent (`drift_share`). Affichez le nombre et la part de colonnes en dérive. Est-ce surprenant, sachant que les jeux d'entraînement et de test d'AMES viennent des mêmes ventes ?

Sauvegardez ensuite ce rapport en JSON, pour un programme, et en HTML, pour un humain.

**Documentation utile**

- https://docs.evidentlyai.com/metrics/customize_data_drift
- https://docs.evidentlyai.com/docs/library/output_formats

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `DataDriftPreset(method="psi", drift_share=0.3)` : `psi` vaut pour les colonnes numériques et catégorielles, alors que `wasserstein` ne s'applique qu'aux numériques (`num_method="wasserstein"`, avec un `cat_method` pour les autres)
- `instantane.dict()["metrics"][0]["value"]` : la première métrique du preset, `DriftedColumnsCount`, avec le nombre (`count`) et la part (`share`) de colonnes en dérive
- `instantane.save_json("rapport-derive.json")`, `instantane.save_html("rapport-derive.html")`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rapport = Report([DataDriftPreset(method=..., drift_share=...)])
instantane = rapport.run(current_data=actuel, reference_data=reference)
print(instantane.dict()["metrics"][0]["value"])
instantane.save_json(...)
instantane.save_html(...)
```

#### Solution

In [ ]:
rapport = Report([DataDriftPreset(method="psi", drift_share=0.3)])
instantane = rapport.run(current_data=actuel, reference_data=reference)
# Aucune colonne ne dérive : les deux jeux sont tirés des mêmes ventes
print(instantane.dict()["metrics"][0]["value"])
instantane.save_json("rapport-derive.json")
instantane.save_html("rapport-derive.html")

### Tests

Les tests sont plus adaptés que la lecture d'un rapport à un contexte automatisé comme la CI/CD. Ils s'attachent aux métriques d'un rapport, par leur paramètre `tests` : une condition (`lt`, `lte`, `gt`, `gte`, `eq`…) sur la valeur de la métrique.

Créez un rapport dont les tests vérifient qu'au plus 5 colonnes dérivent (`DriftedColumnsCount`), et que le score de dérive de `OverallQual` et celui de `GrLivArea` (`ValueDrift`) restent sous 0,05.

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/tests
- https://docs.evidentlyai.com/metrics/all_metrics#data-drift

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `DriftedColumnsCount(tests=[lte(5)])` : un test sur le nombre de colonnes en dérive (`share_tests=[...]` en teste la part)
- `ValueDrift(column="OverallQual", tests=[lt(0.05)])` : un test sur le score de dérive d'une colonne (ici une distance, que la méthode par défaut choisit d'après le type de la colonne)
- `Report([...], include_tests=True)` attacherait à la place leurs tests par défaut aux métriques qui n'en ont pas

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rapport = Report(
  [
    DriftedColumnsCount(tests=[...]),
    ValueDrift(column=..., tests=[...]),
    ValueDrift(column=..., tests=[...]),
  ]
)
instantane = rapport.run(current_data=actuel, reference_data=reference)
instantane
```

#### Solution

In [ ]:
rapport = Report(
  [
    DriftedColumnsCount(tests=[lte(5)]),
    ValueDrift(column="OverallQual", tests=[lt(0.05)]),
    ValueDrift(column="GrLivArea", tests=[lt(0.05)]),
  ]
)
instantane = rapport.run(current_data=actuel, reference_data=reference)
instantane

### Analyse des résultats en Python

Un pipeline lit le résultat des tests en Python. À partir de `instantane.dict()`, calculez le pourcentage de tests réussis du dernier rapport, et affichez le nom des tests échoués.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `instantane.dict()["tests"]` : une liste de dictionnaires, un par test, avec les clés `name`, `description` et `status` (`"SUCCESS"` ou `"FAIL"`)
- sur le rapport précédent, un des trois tests échoue

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
tests = instantane.dict()["tests"]
reussis = sum(test["status"] == ... for test in tests)
print(...)
print([test["name"] for test in tests if ...])
```

#### Solution

In [ ]:
tests = instantane.dict()["tests"]
reussis = sum(test["status"] == "SUCCESS" for test in tests)
print(f"Tests réussis : {reussis / len(tests):.0%}")
print("Tests échoués :", [test["name"] for test in tests if test["status"] == "FAIL"])

## Application de la bibliothèque `evidently` à des données textuelles

### Chargement des données

Les données que nous allons utiliser sont des avis à propos de médicaments traitant plusieurs pathologies. Nous allons pour l'instant nous intéresser aux médicaments traitant la douleur, *Pain* dans les données.

In [ ]:
donnees_brutes = telecharge_avis_medicaments()
avis_douleur = filtre_avis_medicaments(donnees_brutes, "Pain")
reference_df, actuel_df = separe_avis_medicaments(avis_douleur)

In [ ]:
reference_df

### Entraînement d'un modèle de classification

In [ ]:
pipeline = sklearn.pipeline.Pipeline(
  [
    (
      "vectorization",
      sklearn.feature_extraction.text.TfidfVectorizer(
        sublinear_tf=True, max_df=0.5, stop_words="english"
      ),
    ),
    (
      "classification",
      sklearn.linear_model.SGDClassifier(
        alpha=0.0001,
        max_iter=50,
        penalty="l1",
        loss="modified_huber",
        random_state=42,
      ),
    ),
  ]
)
pipeline.fit(reference_df["review"].values, reference_df["est_positif"].values)

Création d'une nouvelle colonne dans `reference_df` & `actuel_df` qui contient les prédictions :

In [ ]:
reference_df["predictions"] = pipeline.predict(reference_df["review"].values)
actuel_df["predictions"] = pipeline.predict(actuel_df["review"].values)

In [ ]:
reference_df

Jusqu'ici, `rapport.run` recevait des `DataFrame`, dont Evidently déduisait le type de chaque colonne. Une colonne de texte et une classification (sa cible et ses prédictions), il faut les lui déclarer : un `Dataset` associe à un `DataFrame` une `DataDefinition`, qui donne le rôle de chaque colonne.

Créez deux `Dataset` (`reference` et `actuel`) avec `Dataset.from_pandas` et une `DataDefinition` adaptée aux trois colonnes `review`, `est_positif` et `predictions` de nos données : une colonne de texte, et une classification binaire (sa cible et ses prédictions).

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/data_definition

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `Dataset.from_pandas(df, data_definition=...)` : un `Dataset` par `DataFrame`
- `DataDefinition(text_columns=[...], classification=[...])`
- `BinaryClassification(target="est_positif", prediction_labels="predictions")` : des prédictions sous forme d'étiquettes (0 ou 1), et non de probabilités

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
definition_donnees = DataDefinition(
  text_columns=[...],
  classification=[BinaryClassification(target=..., prediction_labels=...)],
)
reference = Dataset.from_pandas(reference_df, data_definition=definition_donnees)
actuel = ...
```

#### Solution

In [ ]:
definition_donnees = DataDefinition(
  text_columns=["review"],
  classification=[
    BinaryClassification(target="est_positif", prediction_labels="predictions")
  ],
)

reference = Dataset.from_pandas(reference_df, data_definition=definition_donnees)
actuel = Dataset.from_pandas(actuel_df, data_definition=definition_donnees)

### Détection d'une dérive « technique » des données

Nous allons simuler un événement courant dans une chaîne de traitement : un bug entraîne des traitements de mauvaise qualité. Ici nous allons même en simuler deux :

- Bug dans le nettoyage des balises HTML pendant le prétraitement des avis
- Bug de récupération des données entraînant des données dans une langue différente de la langue d'entraînement

In [ ]:
# La traduction automatique en français de chaque avis sur la douleur : la
# « langue différente » du bug de récupération des données
avis_traduits = pandas.read_csv(
  "https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_pain_fr.tsv",
  sep="\t",
  index_col="id",
)["review"]


balises_html_aleatoires = (
  "<body>",
  "</body>",
  "<html><body>",
  "</body></html>",
  "<h1>",
  "</h1>",
  '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 0 0" width="0" height="0" focusable="false" role="none" style="visibility: hidden; position: absolute; left: -9999px; overflow: hidden;"><defs><filter id="wp-duotone-magenta-yellow"><feColorMatrix color-interpolation-filters="sRGB" type="matrix" values=" .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 "></feColorMatrix><feComponentTransfer color-interpolation-filters="sRGB"><feFuncR type="table" tableValues="0.78039215686275 1"></feFuncR><feFuncG type="table" tableValues="0 0.94901960784314"></feFuncG><feFuncB type="table" tableValues="0.35294117647059 0.47058823529412"></feFuncB><feFuncA type="table" tableValues="1 1"></feFuncA></feComponentTransfer><feComposite in2="SourceGraphic" operator="in"></feComposite></filter></defs></svg>',
)


def injecte_balises_html_aleatoires(texte):
  nb_balises = 25
  for i in range(nb_balises):
    random.seed(i)
    position = random.choice(range(len(texte)))
    texte = texte[:position] + random.choice(balises_html_aleatoires) + texte[position:]

  return texte

In [ ]:
actuel_perturbe_df = actuel_df[["review", "est_positif"]].copy()

In [ ]:
nb_perturbes = int(len(actuel_perturbe_df) * 0.5)
random.seed(42)
indices_perturbes = random.sample(list(actuel_perturbe_df.index), k=nb_perturbes)
actuel_perturbe_df.loc[indices_perturbes[: int(nb_perturbes / 10)], "review"] = (
  actuel_perturbe_df.loc[indices_perturbes[: int(nb_perturbes / 10)], "review"].apply(
    injecte_balises_html_aleatoires
  )
)
traduits = indices_perturbes[int(nb_perturbes / 10) :]
actuel_perturbe_df.loc[traduits, "review"] = avis_traduits.loc[traduits]

In [ ]:
actuel_perturbe_df["predictions"] = pipeline.predict(
  actuel_perturbe_df["review"].values
)
actuel_perturbe = Dataset.from_pandas(
  actuel_perturbe_df, data_definition=definition_donnees
)

### Rapports de qualité de classification

Créez deux rapports de qualité de classification avec le preset `ClassificationPreset` : le premier compare les données de validation « propres » (`actuel`) aux données d'entraînement (`reference`), le second les données détériorées (`actuel_perturbe`) à `actuel`. De combien l'exactitude baisse-t-elle avec les bugs ? Ce rapport aurait-il été disponible le jour du bug, sachant qu'il demande les étiquettes ?

**Documentation utile**

- https://docs.evidentlyai.com/metrics/preset_classification

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `Report([ClassificationPreset()])`, puis un `rapport.run(current_data=..., reference_data=...)` par rapport
- `display(instantane)` affiche un snapshot qui n'est pas la dernière expression de la cellule

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rapport = Report([...])
display(rapport.run(current_data=actuel, reference_data=...))
instantane = rapport.run(current_data=..., reference_data=...)
instantane
```

#### Solution

In [ ]:
rapport = Report([ClassificationPreset()])
display(rapport.run(current_data=actuel, reference_data=reference))
# 100 % d'exactitude sur les avis d'entraînement, que le modèle a appris, 84 % sur
# la validation, 76 % sur les données détériorées : les bugs coûtent 8 points.
# Mais ce rapport demande les étiquettes, qui arrivent bien après les
# prédictions : le jour du bug, il n'aurait pas été disponible. La suite cherche
# un signal dans les seules entrées et prédictions.
instantane = rapport.run(current_data=actuel_perturbe, reference_data=actuel)
instantane

### Analyse des mauvaises performances du modèle

Produisez un rapport de dérive des données limité aux colonnes `est_positif` et `predictions` et aux descripteurs textuels de la colonne `review`, qui compare `actuel_perturbe` à `actuel`.

Définissez d'abord une liste `descripteurs` de descripteurs : la part de caractères qui ne sont pas des lettres, la part de mots hors vocabulaire, le nombre de phrases, la longueur du texte et le nombre de mots. Ajoutez-les à `actuel` et à `actuel_perturbe` avec leur méthode `add_descriptors`.

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/descriptors
- https://docs.evidentlyai.com/metrics/all_descriptors

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `NonLetterCharacterPercentage`, `OOVWordsPercentage`, `SentenceCount`, `TextLength`, `WordCount` : chacun prend la colonne de texte, `"review"`, et un `alias`, le nom de sa colonne ; la suite du notebook utilise les alias `non_letters`, `oov`, `sentence_count`, `text_length` et `word_count`
- `DataDriftPreset(columns=[...])` : la liste des colonnes à comparer, alias des descripteurs compris

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
descripteurs = [
  NonLetterCharacterPercentage("review", alias="non_letters"),
  OOVWordsPercentage("review", alias=...),
  ...,
]
actuel.add_descriptors(descripteurs)
actuel_perturbe.add_descriptors(...)
rapport = Report([DataDriftPreset(columns=["est_positif", "predictions", ...])])
instantane = rapport.run(current_data=actuel_perturbe, reference_data=actuel)
instantane
```

#### Solution

In [ ]:
descripteurs = [
  NonLetterCharacterPercentage("review", alias="non_letters"),
  OOVWordsPercentage("review", alias="oov"),
  SentenceCount("review", alias="sentence_count"),
  TextLength("review", alias="text_length"),
  WordCount("review", alias="word_count"),
]
colonnes_descripteurs = [
  "non_letters",
  "oov",
  "sentence_count",
  "text_length",
  "word_count",
]

actuel.add_descriptors(descripteurs)
actuel_perturbe.add_descriptors(descripteurs)

rapport = Report(
  [DataDriftPreset(columns=["est_positif", "predictions", *colonnes_descripteurs])]
)
instantane = rapport.run(current_data=actuel_perturbe, reference_data=actuel)
instantane

### Inspection manuelle des exemples fautifs

On peut constater l'augmentation des textes longs et de la présence de mots hors vocabulaire (*OOV*).

Utilisez l'export des descripteurs en dataframe pour observer les exemples fautifs, par exemple :

- Les avis qui ont une longueur supérieure à 1000 caractères
- Les avis qui ont plus de 30 % des mots hors du vocabulaire. Comment `evidently` définit-il le vocabulaire ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `actuel_perturbe.as_dataframe()` : les colonnes du `Dataset`, plus une par descripteur, nommée d'après son alias
- pour le vocabulaire, le code d'`OOVWordsPercentage` est dans le paquet installé (`evidently/legacy/features/OOV_words_percentage_feature.py`), et la cellule de mise en place télécharge deux corpus de nltk

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
descripteurs_df = actuel_perturbe.as_dataframe()
descripteurs_df[descripteurs_df["text_length"] > ...]
```

#### Solution

In [ ]:
descripteurs_df = actuel_perturbe.as_dataframe()
descripteurs_df

In [ ]:
descripteurs_df[descripteurs_df["text_length"] > 1000]

In [ ]:
# Le vocabulaire d'evidently est le corpus `words` de nltk (236 736 mots anglais) :
# chaque mot, débarrassé de ce qui n'est ni lettre ni chiffre et mis en minuscules,
# est ramené à son lemme par le lemmatiseur WordNet. Un nombre, un nom de
# médicament ou un mot français est donc hors vocabulaire.
descripteurs_df[descripteurs_df["oov"] > 30]

### Dérive progressive des données

En production, la population change rarement d'un coup. Simulons des lots de 300 avis où la part d'avis sur des médicaments contre la dépression, *Depression* dans les données, monte de 0 à 100 % : le modèle, entraîné sur la douleur, voit arriver des avis d'un autre domaine.

In [ ]:
nouveau_contenu_df = filtre_avis_medicaments(donnees_brutes, "Depression")
nouveau_contenu_df["predictions"] = pipeline.predict(
  nouveau_contenu_df["review"].values
)

reference.add_descriptors(descripteurs)
colonnes_surveillees = ["review", "predictions", *colonnes_descripteurs]


def fabrique_lot(part_depression: float, taille: int = 300) -> Dataset:
  nb_depression = int(taille * part_depression)
  lot_df = pandas.concat(
    [
      actuel_df.sample(taille - nb_depression, random_state=42),
      nouveau_contenu_df.sample(nb_depression, random_state=42),
    ]
  )
  return Dataset.from_pandas(
    lot_df, data_definition=definition_donnees, descriptors=descripteurs
  )

### Lever une alerte

En production, les étiquettes arrivent bien après les prédictions (quand elles arrivent) : l'alerte se lève sur les entrées et les prédictions seules, et reste un signal à analyser avant tout réentraînement. Créez un rapport `rapport_derive` dont les tests vérifient que :

- moins de 30 % des colonnes surveillées (`colonnes_surveillees`) dérivent ;
- le score de dérive du texte de la colonne `review` reste sous 0,6.

Écrivez ensuite `leve_alerte(instantane)`, qui renvoie `True` si l'un des tests du snapshot a échoué. Essayez-la sur `fabrique_lot(0)` et sur `fabrique_lot(1)`, face aux données d'entraînement (`reference`), la référence du monitoring en production.

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/tests

In [ ]:
# Votre code ici


def leve_alerte(instantane) -> bool: ...

#### Conseils supplémentaires

- `DriftedColumnsCount(columns=colonnes_surveillees, share_tests=[lt(0.3)])` : un test sur la part de colonnes en dérive, et non sur leur nombre
- `ValueDrift(column="review", tests=[lt(0.6)])` : sur une colonne de texte, evidently entraîne un classifieur à distinguer les textes récents de ceux de référence ; son score est l'AUC ROC de ce classifieur (0,5 : indiscernables)
- `instantane.dict()["tests"]` et leur `status`, comme plus haut ; sur `fabrique_lot(0)`, `leve_alerte` renvoie `False`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rapport_derive = Report(
  [
    DriftedColumnsCount(columns=..., share_tests=[...]),
    ValueDrift(column="review", tests=[...]),
  ]
)


def leve_alerte(instantane) -> bool:
  return any(test["status"] == ... for test in instantane.dict()["tests"])


for part in [0, 1]:
  instantane = rapport_derive.run(
    current_data=fabrique_lot(part), reference_data=reference
  )
  print(leve_alerte(instantane))
```

#### Solution

In [ ]:
rapport_derive = Report(
  [
    DriftedColumnsCount(columns=colonnes_surveillees, share_tests=[lt(0.3)]),
    ValueDrift(column="review", tests=[lt(0.6)]),
  ]
)


def leve_alerte(instantane) -> bool:
  return any(test["status"] == "FAIL" for test in instantane.dict()["tests"])


for part in [0, 1]:
  instantane = rapport_derive.run(
    current_data=fabrique_lot(part), reference_data=reference
  )
  print(leve_alerte(instantane))

### Tableau de bord

Un tableau de bord suit ces indicateurs lot après lot. Pour chaque part d'avis *Depression* de `[0, 0.2, 0.4, 0.6, 0.8, 1]`, lancez `rapport_derive` sur `fabrique_lot(part)` face à `reference`, et rangez dans un `DataFrame` `tableau_de_bord`, une ligne par lot :

- la part d'avis *Depression* ;
- la part de colonnes en dérive et le score de dérive de `review` ;
- l'alerte de `leve_alerte` ;
- l'exactitude du modèle sur le lot, calculée avec les étiquettes, qui en production n'arriveraient que bien plus tard.

Tracez la part de colonnes en dérive, le score de `review` et l'exactitude en fonction de la part d'avis *Depression*. L'alerte aurait-elle été levée à temps, avant que l'exactitude ne baisse vraiment ?

**Documentation utile**

- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `instantane.dict()["metrics"]` : les métriques de `rapport_derive` dans leur ordre ; `[0]["value"]["share"]` est la part de colonnes en dérive, `[1]["value"]` le score de `review`
- `lot.as_dataframe()` : les colonnes `predictions` et `est_positif` du lot, dont l'égalité moyenne est l'exactitude
- `tableau_de_bord.plot(x="part_depression", y=[...], marker="o")`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
lignes = []
for part in [0, 0.2, 0.4, 0.6, 0.8, 1]:
  lot = fabrique_lot(part)
  instantane = rapport_derive.run(current_data=lot, reference_data=reference)
  metriques = instantane.dict()["metrics"]
  lot_df = lot.as_dataframe()
  lignes.append(
    {
      "part_depression": part,
      "colonnes_derivees": ...,
      "derive_avis": ...,
      "alerte": leve_alerte(instantane),
      "exactitude": (lot_df["predictions"] == ...).mean(),
    }
  )
tableau_de_bord = pandas.DataFrame(lignes)
tableau_de_bord.plot(x="part_depression", y=[...], marker="o")
tableau_de_bord
```

#### Solution

In [ ]:
lignes = []
for part in [0, 0.2, 0.4, 0.6, 0.8, 1]:
  lot = fabrique_lot(part)
  instantane = rapport_derive.run(current_data=lot, reference_data=reference)
  metriques = instantane.dict()["metrics"]
  lot_df = lot.as_dataframe()
  lignes.append(
    {
      "part_depression": part,
      "colonnes_derivees": metriques[0]["value"]["share"],
      "derive_avis": metriques[1]["value"],
      "alerte": leve_alerte(instantane),
      "exactitude": (lot_df["predictions"] == lot_df["est_positif"]).mean(),
    }
  )
tableau_de_bord = pandas.DataFrame(lignes)
tableau_de_bord.plot(
  x="part_depression",
  y=["colonnes_derivees", "derive_avis", "exactitude"],
  marker="o",
)
# Le texte des avis trahit le changement de domaine sans attendre les étiquettes :
# l'alerte est levée alors que l'exactitude n'a encore que peu baissé : à analyser
# avant tout réentraînement.
premier = tableau_de_bord.loc[tableau_de_bord["alerte"], "part_depression"].min()
print(f"Alerte levée dès {premier:.0%} d'avis Depression")
tableau_de_bord

### Dérive des concepts

Dans une dérive des concepts, les entrées ne changent pas : c'est leur relation avec la cible, $P(y \mid x)$, qui change, et les mêmes avis ne veulent plus dire la même chose. Simulons-la sur les données de validation : les avis qui contiennent « work » (*works*, *worked*…) voient leur étiquette inversée, comme si le sens de ce mot s'était retourné pour les patients.

In [ ]:
concept_df = actuel_df.copy()
inverses = concept_df["review"].str.contains("work", case=False)
concept_df.loc[inverses, "est_positif"] = 1 - concept_df.loc[inverses, "est_positif"]
concept = Dataset.from_pandas(
  concept_df, data_definition=definition_donnees, descriptors=descripteurs
)
print(f"Étiquettes inversées : {inverses.mean():.0%} des avis")

### Détecter une dérive des concepts

Lancez `rapport_derive` sur `concept` face à `reference`, comme pour les lots : `leve_alerte` lève-t-elle une alerte ? Produisez ensuite un rapport de qualité de classification qui compare `concept` aux mêmes avis avant la dérive, `actuel`. Quel signal révèle cette dérive, et quand serait-il disponible en production ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `rapport_derive.run(current_data=concept, reference_data=reference)` et `leve_alerte`, comme pour les lots
- `Report([ClassificationPreset()])` sur `concept` et `actuel`, comme plus haut

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(leve_alerte(rapport_derive.run(current_data=..., reference_data=...)))
rapport = Report([ClassificationPreset()])
instantane = rapport.run(current_data=..., reference_data=...)
instantane
```

#### Solution

In [ ]:
# Les entrées et les prédictions sont celles de `actuel`, qui suivent la même
# distribution que les données d'entraînement : les tests de dérive passent. Seule
# la qualité de la classification, mesurée une fois les étiquettes connues, révèle
# la dérive des concepts.
instantane_derive = rapport_derive.run(current_data=concept, reference_data=reference)
print(f"leve_alerte : {leve_alerte(instantane_derive)}")
rapport = Report([ClassificationPreset()])
instantane = rapport.run(current_data=concept, reference_data=actuel)
instantane